# Local Gemma evaluation

This notebook runs the published `swegemma` evaluator on the public task snapshots. It is evaluation, not LoRA training.


In [ ]:
from pathlib import Path
import os, subprocess, sys

PROJECT = Path.cwd()
while PROJECT != PROJECT.parent and not (PROJECT / 'scripts' / 'run_local_eval.py').exists():
    PROJECT = PROJECT.parent
if not (PROJECT / 'scripts' / 'run_local_eval.py').exists():
    raise FileNotFoundError('Open this notebook inside Denisha_Gemma4_Agents or set PROJECT manually.')
os.chdir(PROJECT)
print('Project:', PROJECT)


In [ ]:
# Build a fresh integrated submission before evaluation.
subprocess.run([sys.executable, 'scripts/smoke_test.py', '--variant', 'agent_2_integrated'], check=True)
subprocess.run([sys.executable, 'scripts/build_submission.py', '--variant', 'agent_2_integrated'], check=True)

# Change DATA_ROOT to the folder containing tasks.jsonl, snapshots/, and sample_submission/.
DATA_ROOT = PROJECT / 'data'
TASKS = DATA_ROOT / 'tasks.jsonl'
SNAPSHOTS = DATA_ROOT / 'snapshots'
REFERENCE_SUBMISSION = DATA_ROOT / 'sample_submission'
SUBMISSION = PROJECT / 'build' / 'agent_2_integrated'
RESULTS = PROJECT / 'results' / 'denisha_local'
for label, path in [('tasks.jsonl', TASKS), ('snapshots/', SNAPSHOTS), ('sample_submission/', REFERENCE_SUBMISSION), ('submission/', SUBMISSION)]:
    print(label, path, 'OK' if path.exists() else 'MISSING')
if not all(p.exists() for p in (TASKS, SNAPSHOTS, REFERENCE_SUBMISSION, SUBMISSION)):
    raise FileNotFoundError('Place tasks.jsonl, snapshots/, and sample_submission/ under data/ or update DATA_ROOT before evaluation.')


In [ ]:
subprocess.run([sys.executable, 'scripts/run_local_eval.py', '--tasks', str(TASKS), '--snapshots-dir', str(SNAPSHOTS), '--submission-dir', str(SUBMISSION), '--results-dir', str(RESULTS), '--sandbox', 'docker', '--max-tool-calls', '50', '--max-time-minutes', '30', '--concurrency', '1'], check=True)


In [ ]:
summary = RESULTS / 'summary.json'
print(summary)
if summary.exists():
    print(summary.read_text())
